# Correlation matrices + rates/asset scatter

1. **Level corr** — `CORR_SERIES`
2. **Corr percentile** — 100 = más alto en `CORR_PCT_LOOKBACK`
3. **Scatter** — x = ρ(USGG5YR, CO1), y = ρ(USGG5YR, ESA); últimos 4 días en azul

`run_correlations()` **descarga solos** los tickers de `CROSS_ASSET` / USSFCT que falten en `sofr_live.csv`.

**Importante:** unpack de 5 valores:
```python
level, pmat, full, scatter_path, scatter_df = correlations.run_correlations()
```

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "correlations.py").exists() else _DZ
if not (REPO / "correlations.py").exists():
    raise FileNotFoundError("No encuentro correlations.py — git pull origin main")

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
import curve_config
import correlations
import data

importlib.reload(curve_config)
importlib.reload(data)
importlib.reload(correlations)

print("CROSS_ASSET:", curve_config.CROSS_ASSET)
print("CORR_SERIES:", curve_config.CORR_SERIES)
print("window=%s  pct_lookback=%s" % (
    curve_config.CORR_WINDOW, curve_config.CORR_PCT_LOOKBACK,
))

In [ ]:
# 5 valores (NO uses level, zmat, full = ...)
level, pmat, full, scatter_path, scatter_df = correlations.run_correlations()

uss = [c for c in full.columns if str(c).upper().startswith("USSFCT") or c in ("CO1", "SPX", "ESA", "USGG5YR")]
print("asof:", full.index[-1], "n=", len(full))
print("cross / USSFCT in panel:", uss)
print("scatter:", scatter_path)
if not uss:
    print("WARNING: still no CO1/USSFCT — BBG pull failed; try: python run_dz.py prediction --once --pull-history")

In [ ]:
# ── 1) Matriz de correlaciones (nivel) ───────────────────────────────────────
display(level.round(3))

In [ ]:
# ── 2) Percentil del rolling correlation (100 = máximo) ─────────────────────
display(pmat.round(1))
heat = Path(curve_config.GRAPH_DIR_CORR) / "corr_percentile_heatmap.png"
display(Image(filename=str(heat))) if heat.exists() else print("no heatmap")

In [ ]:
# Level heatmap
heat_l = Path(curve_config.GRAPH_DIR_CORR) / "corr_level_heatmap.png"
display(Image(filename=str(heat_l))) if heat_l.exists() else None

In [ ]:
# ── 3) Scatter ρ(rates, CO1) vs ρ(rates, ESA) ────────────────────────────────
heat_s = Path(curve_config.GRAPH_DIR_CORR) / "corr_scatter_rates_assets.png"
if heat_s.exists():
    display(Image(filename=str(heat_s)))
elif scatter_path and Path(scatter_path).exists():
    display(Image(filename=str(scatter_path)))
else:
    print("no scatter yet")
if scatter_df is not None:
    display(scatter_df.tail(10))